# <font color="steelblue">25. Población, muestra y tipos de muestreo</font>

*Unidad 5. Conceptos básicos de inferencia estadística* · Sesión teórica 25 de 30

**Autoría**:

*   Fernando Borrás Rocher (f.borras@umh.es), autor de la idea original
*   Juan Francisco Monge Ivars
*   José Santiago García Cremades

Departamento de Estadística, Matemáticas e Informática. Universidad Miguel Hernández de Elche.

Basado en [SIMUSTAT](https://unidigitalsimustat.umh.es/) (proyecto UNIDIGITAL-UMH, CC BY-SA 4.0).

**Asignatura**: Estadística (1200), 1.º curso, Grado en Periodismo.

**Licencia**: <a rel="license" href="http://creativecommons.org/licenses/by-sa/4.0/"><img alt="CC BY-SA 4.0" style="border-width:0" src="https://i.creativecommons.org/l/by-sa/4.0/88x31.png" /></a> Creative Commons Atribución-CompartirIgual 4.0.

> ⚠️ **Los datos de este curso son simulados.** Imitan la forma de fuentes reales (INE, CIS, audímetros, agencias de verificación), pero ninguna cifra es real. Úsalos para aprender; para publicar, ve siempre a la fuente oficial.

## <font color='steelblue'>Antes de empezar: configura el cuaderno</font>

Ejecuta **una vez** la celda de abajo (botón ▶). Instala la librería de las preguntas, descarga los datos (`encuesta_medios.csv`) y las preguntas de autoevaluación. Si todo va bien verás **`Cuaderno configurado`**.

> 💡 Si Colab se desconecta, los ficheros se borran: vuelve a ejecutar esta celda.

In [ ]:
#@title <font color="steelblue" size="+1"> Ejecutar configuración { display-mode: "form" }
import os, sys, json, base64, subprocess, urllib.request

RAW = "https://raw.githubusercontent.com/fborrasumh/statperiodismo/main/"
DATOS = ["encuesta_medios.csv"]
PREGUNTAS = ["q_P25_1.json", "q_P25_2.json", "q_P25_3.json", "q_P25_4.json"]

try:
    from jupyterquiz import display_quiz
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jupyterquiz==2.9.6.4"], check=True)
    from jupyterquiz import display_quiz

for carpeta, ficheros in (("datos/", DATOS), ("autoeval/", PREGUNTAS)):
    for f in ficheros:
        if not os.path.exists(f):
            urllib.request.urlretrieve(RAW + carpeta + f, f)

def pregunta(fichero):
    """Muestra una pregunta interactiva de autoevaluación."""
    display_quiz(json.loads(base64.b64decode(open(fichero, "rb").read()).decode("utf8")))

print("Cuaderno configurado")

## <font color='steelblue'>Introducción</font>

Ningún instituto pregunta a todo el país: entrevista a 1.000 o 2.000 personas y de ahí saca conclusiones para millones. Que eso funcione depende de **cómo** se elige a esas personas. Una encuesta de 1.000 entrevistas bien hecha vale mucho más que una votación en redes con 50.000 respuestas. Este cuaderno explica por qué.

**Qué necesitas saber antes**: la unidad 4, en especial los cuadernos 20 y 23.

## <font color='steelblue'>Qué vas a aprender</font>

Al terminar este cuaderno sabrás:

1. Distinguir población, muestra, parámetro y estadístico.
2. Describir los principales tipos de muestreo y cuándo se usa cada uno.
3. Reconocer los sesgos de selección, de no respuesta y de autoselección.
4. Entender para qué sirve ponderar una muestra.

## <font color='steelblue'>Contenidos</font>

1. Población, muestra, parámetro y estadístico
2. Muestreo aleatorio simple y estratificado
3. Sesgos: cuando la muestra no se parece a la población
4. Ponderar una muestra

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (10, 4)
pd.set_option('display.max_columns', 20)

## <font color='steelblue'>1. Población, muestra, parámetro y estadístico</font>

| Concepto | Qué es | Ejemplo |
| :-- | :-- | :-- |
| **Población** | Todo el grupo sobre el que se quiere saber algo | Residentes mayores de 18 años en la Comunitat Valenciana |
| **Censo** | Estudiar a toda la población | Las elecciones son un censo de los votantes |
| **Muestra** | Parte de la población que se estudia | 1.200 personas entrevistadas |
| **Parámetro** | Valor (desconocido) en la población | % real que se informa por redes |
| **Estadístico** | Valor calculado en la muestra | % de los entrevistados que se informa por redes |

La **inferencia** consiste en usar el estadístico para decir algo del parámetro, **con una medida de su error**.

**Qué vamos a hacer**: para poder comprobar lo que hacemos, en este cuaderno trataremos las 1.200 personas de la encuesta como si fueran **toda la población**. Así conocemos el «parámetro» y podemos ver cuánto se equivocan distintas muestras.

In [ ]:
poblacion = pd.read_csv("encuesta_medios.csv")
parametro = (poblacion["medio_principal"] == "Redes sociales").mean()
print(f"Parámetro (en la 'población'): {parametro:.3f}")

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y escoge la respuesta correcta (**solo hay una**):

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P25_1.json")

## <font color='steelblue'>2. Muestreo aleatorio simple y estratificado</font>

En el **muestreo aleatorio simple** (MAS) cada persona tiene la misma probabilidad de ser elegida. Con Pandas: `sample(n)`.

In [ ]:
rng = np.random.default_rng(25)
muestra = poblacion.sample(100, random_state=1)
print(f"Estadístico en una muestra de 100: {(muestra['medio_principal'] == 'Redes sociales').mean():.3f}")

estimaciones = [(poblacion.sample(100, random_state=i)["medio_principal"] == "Redes sociales").mean() for i in range(5)]
print("Cinco muestras distintas:", np.round(estimaciones, 3))

Cada muestra da un resultado distinto: es el **error muestral**, que veremos en el cuaderno 26.

En el **muestreo estratificado** la población se divide en grupos (**estratos**: provincias, grupos de edad) y se extrae una muestra de cada uno. Garantiza que todos los grupos estén representados y suele reducir el error. Es lo que hacen el CIS y casi todos los institutos.

In [ ]:
prop = poblacion["provincia"].value_counts(normalize=True)
n_total = 100
estratificada = pd.concat([poblacion[poblacion["provincia"] == prov].sample(round(n_total * p), random_state=1)
                           for prov, p in prop.items()])
print(estratificada["provincia"].value_counts())

Otros tipos que verás en las fichas técnicas:

- **Por conglomerados**: se eligen al azar grupos enteros (municipios, secciones censales) y dentro de ellos a las personas. Abarata el trabajo de campo.
- **Por cuotas**: se fijan cuántas personas de cada sexo y edad hay que entrevistar, pero la elección final no es aleatoria.
- **Polietápico**: combina varios (primero municipios, luego hogares, luego personas).

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y escoge la respuesta correcta (**solo hay una**):

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P25_2.json")

## <font color='steelblue'>3. Sesgos: cuando la muestra no se parece a la población</font>

Un **sesgo** es un error que va siempre en la misma dirección y **no disminuye aunque aumente la muestra**. Los más frecuentes:

| Sesgo | Qué pasa | Ejemplo |
| :-- | :-- | :-- |
| **De selección** | Algunos grupos tienen más probabilidad de entrar | Encuesta solo por teléfono fijo: faltan jóvenes |
| **De autoselección** | Las personas deciden si participan | Votaciones en redes o en la web del diario |
| **De no respuesta** | Quienes no contestan son distintos | Los más desconfiados no responden |
| **De deseabilidad social** | Se responde lo que «queda bien» | Se declara más lectura de la que hay |

**Qué vamos a hacer**: simular una encuesta en redes sociales. Solo responde quien se informa por redes (que participa mucho más) o quien comparte noticias. Aunque recojamos muchas respuestas, el resultado está sesgado.

In [ ]:
p_participar = np.where(poblacion["medio_principal"] == "Redes sociales", 0.6, 0.05)
p_participar = p_participar + np.where(poblacion["comparte_noticias"] == "Sí", 0.1, 0)
responde = rng.random(len(poblacion)) < p_participar
encuesta_redes = poblacion[responde]
print("Respuestas recogidas:", len(encuesta_redes))
print(f"% que se informa por redes según la encuesta en redes: {(encuesta_redes['medio_principal'] == 'Redes sociales').mean():.1%}")
print(f"Valor real (parámetro): {parametro:.1%}")

**Qué ha pasado**: la «encuesta» en redes multiplica por más de tres el porcentaje real. Más respuestas no lo arreglan: el problema es **quién** responde. Por eso una votación en redes **nunca** es una encuesta, y no debe publicarse como tal.

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y escoge la respuesta correcta (**solo hay una**):

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P25_3.json")

## <font color='steelblue'>4. Ponderar una muestra</font>

Si la muestra tiene, por ejemplo, demasiados universitarios, se puede **ponderar**: dar a cada persona un peso igual a (proporción en la población) / (proporción en la muestra). Así los grupos poco representados cuentan más. Corrige desequilibrios **conocidos**, pero no los sesgos de grupos que ni siquiera están en la muestra.

In [ ]:
# Muestra con demasiados universitarios
rng_p = np.random.default_rng(1)
p_entrar = np.where(poblacion["estudios"] == "Universitarios", 0.9, 0.3)
sesgada = poblacion[rng_p.random(len(poblacion)) < p_entrar].copy()

prop_pob = poblacion["estudios"].value_counts(normalize=True)
prop_mue = sesgada["estudios"].value_counts(normalize=True)
sesgada["peso"] = sesgada["estudios"].map(prop_pob / prop_mue)

real = (poblacion["suscripcion_pago"] == "Sí").mean()
sin_pesos = (sesgada["suscripcion_pago"] == "Sí").mean()
con_pesos = np.average(sesgada["suscripcion_pago"] == "Sí", weights=sesgada["peso"])
print(f"% que paga por noticias. Real: {real:.1%}   Muestra sin ponderar: {sin_pesos:.1%}   Ponderada: {con_pesos:.1%}")

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y escoge la respuesta correcta (**solo hay una**):

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P25_4.json")

## <font color='steelblue'>Reto aprendiz</font>

Resuelve estos ejercicios en la celda vacía de debajo de cada enunciado. Si te atascas, vuelve a la sección indicada. La solución está plegada: **inténtalo antes de mirarla**.

### **Ejercicio 1** *(sección 1)*

Calcula en la «población» el parámetro «% que paga por noticias» y estímalo con una muestra aleatoria simple de 200 personas.

<details><summary>Ver solución</summary>

```python
print((poblacion["suscripcion_pago"] == "Sí").mean())
print((poblacion.sample(200, random_state=7)["suscripcion_pago"] == "Sí").mean())
```
</details>

### **Ejercicio 2** *(sección 2)*

Extrae una muestra estratificada de 120 personas por grupo de estudios, proporcional al tamaño de cada grupo.

<details><summary>Ver solución</summary>

```python
prop = poblacion["estudios"].value_counts(normalize=True)
pd.concat([poblacion[poblacion["estudios"] == e].sample(round(120 * p), random_state=2) for e, p in prop.items()])["estudios"].value_counts()
```
</details>

### **Ejercicio 3** *(sección 3)*

Busca la ficha técnica de una encuesta publicada este mes (CIS u otro instituto). Anota: universo, tamaño de muestra, tipo de muestreo, método de entrevista y fechas.

<details><summary>Ver idea clave</summary>

Ejercicio abierto. Si una noticia sobre una encuesta no da estos datos, el lector no puede valorar su calidad.
</details>

### **Ejercicio 4** *(sección 3)*

Repite la simulación de la encuesta en redes con probabilidades de participación de 0,3 y 0,05. ¿Sigue habiendo sesgo?

<details><summary>Ver solución</summary>

```python
r = rng.random(len(poblacion)) < np.where(poblacion["medio_principal"] == "Redes sociales", 0.3, 0.05)
(poblacion[r]["medio_principal"] == "Redes sociales").mean()
```
</details>

## <font color='steelblue'>Resumen: ideas clave</font>

- La **inferencia** usa un **estadístico** de una **muestra** para conocer un **parámetro** de la **población**.
- **Aleatorio simple**: todos con la misma probabilidad. **Estratificado**: una muestra por grupo. **Conglomerados**: grupos enteros al azar. **Cuotas**: no aleatorio.
- Un **sesgo** no se corrige aumentando la muestra; las votaciones en redes tienen **sesgo de autoselección**.
- **Ponderar** corrige desequilibrios conocidos de la muestra, no los grupos ausentes.
- Exige siempre la **ficha técnica**: universo, tamaño, tipo de muestreo, método y fechas.

## <font color='steelblue'>Glosario</font>

| Término | Significado |
| :-- | :-- |
| **Población** | Conjunto completo sobre el que se quiere saber algo. |
| **Muestra** | Parte de la población que se observa. |
| **Parámetro** | Valor en la población, normalmente desconocido. |
| **Estadístico** | Valor calculado con la muestra. |
| **Sesgo de autoselección** | Error que aparece cuando los participantes deciden ellos mismos si participan. |
| **Ficha técnica** | Descripción de cómo se ha hecho una encuesta. |

## <font color='steelblue'>Errores frecuentes y cómo resolverlos</font>

| Síntoma | Causa probable | Solución |
| :-- | :-- | :-- |
| `NameError: name 'pd' is not defined` | No se ejecutó la celda de importación | Ejecuta las celdas desde el principio, en orden |
| `FileNotFoundError` al leer un fichero | No se ejecutó la configuración o Colab se reinició | Ejecuta la celda *Configura el cuaderno* |
| La pregunta de autoevaluación no aparece | Falta la configuración | Ejecuta la celda de configuración y luego la pregunta |

## <font color='steelblue'>Para seguir</font>

En el cuaderno 26 verás cuánto se equivoca una muestra por puro azar: la distribución muestral y el error estadístico.

**Siguiente cuaderno**: [26. Distribución muestral y error estadístico (simulación de Monte Carlo)](https://colab.research.google.com/github/fborrasumh/statperiodismo/blob/main/cuadernos/P26_distribucion_muestral_y_error.ipynb)